In [ ]:
from common.setup_plotting import setup_matplotlib, get_figure_dir
from common.data_downloader import download_data
from common.training import train_model, load_checkpoint, evaluate
from helper import collate_fn_gnn, GNNEncoder

import numpy as np
import awkward as ak
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, r2_score

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.nn import DynamicEdgeConv, global_mean_pool


setup_matplotlib()        # configure matplotlib first (So i can use LaTeX in the labels)
# Make interactive plots work in Jupyter notebooks 
%matplotlib inline  
import matplotlib.pyplot as plt   # THEN import pyplot
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.colors import PowerNorm

In [ ]:
# download data for this task, if needed and get path
### IF AN ERROR OCCURS HERE, THEN READ THE ERROR MESSAGE BECAUSE IT IS CODED IN 
data_path = download_data("task_04")
# get and if needed create figure directory for this task
fig_dir = get_figure_dir("task_04")

In [ ]:
train_dataset = ak.from_parquet(f"{data_path}/train.pq")
val_dataset = ak.from_parquet(f"{data_path}/val.pq")
test_dataset = ak.from_parquet(f"{data_path}/test.pq")
print("Dataset sizes (train, val, test):",
      len(train_dataset), len(val_dataset), len(test_dataset))
print("Units x position, y position, time: m, m, ns")

In [ ]:
# to get familiar with the dataset, let's inspect it.
print(f"The training dataset contains {len(train_dataset)} events.")
print(f"The validation dataset contains {len(val_dataset)} events.")
print(f"The test dataset contains {len(test_dataset)} events.")
print(
    f"The training dataset has the following columns: {train_dataset.fields}")
print(
    f"The validation dataset has the following columns: {val_dataset.fields}")
print(f"The test dataset has the following columns: {test_dataset.fields}")
# print the first event of the training dataset
print(f"The first event of the training dataset is: {train_dataset[0]}")

# We are interested in the labels xpos and ypos. This is the position of the neutrino interaction that we want to predict.
print(
    f"The first event of the training dataset has the following labels: {train_dataset['xpos'][0]}, {train_dataset['ypos'][0]}")
# Awkward arrays also allow us to obtain the 'xpos' and 'ypos' label for all events in the dataset
print(
    f"The first 10 labels of the training dataset are: {train_dataset['xpos'][:10]}, {train_dataset['ypos'][:10]}")

# The data can be accessed by using the 'data' key.
# The data is a 3D array with the first dimension being the number of events,
# the second dimension being the the three features (time, x, y)
# the third dimension being the number of hits,
print(
    f"The first event of the training dataset has {len(train_dataset['data'][0][0])} hits, i.e., detected photons.")
# Let's loop over all hits and print the time, x, and y coordinates of the first event.
for i in range(len(train_dataset['data'][0, 0])):
    print(
        f"Hit {i}: time = {train_dataset['data'][0,0,i]}, x = {train_dataset['data'][0,1, i]}, y = {train_dataset['data'][0,2,i]}")
# To get all hit times of the first event, you can use the following code:
print(
    f"The first event of the training dataset has the following hit times: {train_dataset['data'][0, 0]}")
print(
    f"The first event of the training dataset has the following hit x positions: {train_dataset['data'][0, 1]}")
print(
    f"The first event of the training dataset has the following hit y positions: {train_dataset['data'][0, 2]}")

In [ ]:
# Calculate normalization constants from the training set only

time_mean = ak.mean(train_dataset["data"][:, 0, :])
time_std = ak.std(train_dataset["data"][:, 0, :])

x_mean = ak.mean(train_dataset["data"][:, 1, :])
x_std = ak.std(train_dataset["data"][:, 1, :])

y_mean = ak.mean(train_dataset["data"][:, 2, :])
y_std = ak.std(train_dataset["data"][:, 2, :])

xpos_mean = ak.mean(train_dataset["xpos"])
xpos_std = ak.std(train_dataset["xpos"])

ypos_mean = ak.mean(train_dataset["ypos"])
ypos_std = ak.std(train_dataset["ypos"])

# Define a function to normalize the data and labels using the calculated normalization constants. This function will be applied to the training, validation, and test datasets.
def normalize_data_and_labels(dataset):
    # Normalize data and labels
    # working with Awkward arrays is a bit tricky because the ['data'] field can't be assigned in-place,
    # so we need to extract the time, x, and y coordinates, normalize them separately,
    # and then concatenate them back together.
    # important to index the time dimension with 0:1 to keep this dimension (n_events, 1, n_hits)
    times = dataset["data"][:, 0:1, :]
    norm_times = (times - time_mean) / time_std

    x = dataset["data"][:, 1:2, :]
    norm_x = (x - x_mean) / x_std

    y = dataset["data"][:, 2:3, :]
    norm_y = (y - y_mean) / y_std

    dataset["data"] = ak.concatenate(
        [norm_times, norm_x, norm_y],
        axis=1
    )

    dataset["xpos"] = (dataset["xpos"] - xpos_mean) / xpos_std
    dataset["ypos"] = (dataset["ypos"] - ypos_mean) / ypos_std

    return dataset

In [ ]:
train_dataset = normalize_data_and_labels(train_dataset)
val_dataset = normalize_data_and_labels(val_dataset)
test_dataset = normalize_data_and_labels(test_dataset)

In [ ]:
batch_size = 64

g = torch.Generator()
g.manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn_gnn,
    generator=g,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn_gnn,
    generator=g
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn_gnn,
    generator=g
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GNNEncoder(k=8).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()


In [ ]:
# Note: if training was skipped the model will load the best model and the training and validation losses from the checkpoint. 
# So after training a model set skip_training to True
skip_training = True
model_path = "../models/gnn_dynamic_edgeconv.pt"


train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    num_epochs=30,
    checkpoint_path=model_path,
    patience=6,
    verbose=True,
    skip_training=skip_training,
)

In [ ]:
checkpoint = load_checkpoint(
    model_path,
    device=device,
)
print("Best validation loss:", checkpoint["best_val_loss"])
print("Best epoch:", checkpoint["epoch"])

# Switch to evaluation mode 
test_loss = evaluate(
    model=model,
    data_loader=test_loader,
    loss_fn=loss_fn,
    device=device,
)
print(f"Test MSE loss: {test_loss:.4f}")

In [ ]:
# Collect all predictions and labels for the test set to calculate additional metrics and make plots

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch_graph, batch_labels in test_loader:
        batch_graph = batch_graph.to(device)
        batch_labels = batch_labels.to(device)

        preds = model(batch_graph)

        all_preds.append(preds.cpu())
        all_labels.append(batch_labels.cpu())

all_preds = torch.cat(all_preds, dim=0)
all_labels = torch.cat(all_labels, dim=0)

print(all_preds.shape)
print(all_labels.shape)

### Now some Evaluation plots

In [ ]:
# Train and validation loss curves
fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(train_losses, label="Train Loss (Train Mode)", color="C0")
ax.plot(val_losses, label="Validation Loss", color="C1")

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Training and Validation Loss Curves")

ax.axhline(
    test_loss,
    color="C2",
    linestyle="--",
    label=f"Test Loss: {test_loss:.4f}",
)

ax.axvline(
    checkpoint["epoch"] - 1, # checkpoint epoch is 1-based, but x-axis is 0-based
    color="black",
    linestyle=":",
    label=f"Best Checkpoint Epoch: {checkpoint['epoch']}",
)

ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/loss_curves.pdf")
None

In [ ]:
# Denormalize predictions and labels to get the actual positions (instead of the normalized values) for the following plots and metrics. 
# This is important for the interpretation of the results. 
pred_x = all_preds[:, 0] * xpos_std + xpos_mean
pred_y = all_preds[:, 1] * ypos_std + ypos_mean

true_x = all_labels[:, 0] * xpos_std + xpos_mean
true_y = all_labels[:, 1] * ypos_std + ypos_mean

# Compute position errors
error_x = pred_x - true_x
error_y = pred_y - true_y

position_error = torch.sqrt(error_x**2 + error_y**2)

mae_x = torch.mean(torch.abs(error_x))
mae_y = torch.mean(torch.abs(error_y))

mean_position_error = torch.mean(position_error)
median_position_error = torch.median(position_error)

print(f"MAE x: {mae_x:.4f}")
print(f"MAE y: {mae_y:.4f}")
print(f"Mean position error: {mean_position_error:.4f}")
print(f"Median position error: {median_position_error:.4f}")


unique_energies = np.unique(ak.to_numpy(test_dataset["energy"]))
print("Unique energies:", unique_energies)
print("Number of unique energies:", len(unique_energies))

In [ ]:
# Predicted vs true x position
fig, ax = plt.subplots(figsize=(4, 3))

ax.scatter(
    true_x.numpy(),
    pred_x.numpy(),
    s=6,
    alpha=0.4,
    color="C0",
    label="Test events",
)

min_val = min(true_x.min().item(), pred_x.min().item())
max_val = max(true_x.max().item(), pred_x.max().item())

ax.plot(
    [min_val, max_val],
    [min_val, max_val],
    color="black",
    linestyle=":",
    label="Perfect prediction",
)

ax.set_xlabel(r"True x position $[\mathrm{m}]$")
ax.set_ylabel(r"Predicted x position $[\mathrm{m}]$")
ax.set_title("Predicted vs True x Position")

ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/pred_vs_true_x.pdf")
None

In [ ]:
# Predicted vs true y position
fig, ax = plt.subplots(figsize=(4, 3))

ax.scatter(
    true_y.numpy(),
    pred_y.numpy(),
    s=6,
    alpha=0.4,
    color="C1",
    label="Test events",
)

min_val = min(true_y.min().item(), pred_y.min().item())
max_val = max(true_y.max().item(), pred_y.max().item())

ax.plot(
    [min_val, max_val],
    [min_val, max_val],
    color="black",
    linestyle=":",
    label="Perfect prediction",
)

ax.set_xlabel(r"True y position $[\mathrm{m}]$")
ax.set_ylabel(r"Predicted y position $[\mathrm{m}]$")
ax.set_title("Predicted vs True y Position")

ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/pred_vs_true_y.pdf")
None

In [ ]:
# Position error statistics
mean_position_error = torch.mean(position_error)
median_position_error = torch.median(position_error)
position_error_std = torch.std(position_error)

# uncertainty on the mean
position_error_sem = position_error_std / torch.sqrt(
    torch.tensor(len(position_error), dtype=torch.float32)
)

# Position error histogram
fig, ax = plt.subplots(figsize=(6, 4))

ax.hist(
    position_error.numpy(),
    bins=50,
    color="C0",
    alpha=0.8,
)

ax.axvline(
    mean_position_error.item(),
    color="C1",
    linestyle="--",
    label=rf"Mean: ${mean_position_error.item():.3f}\,\mathrm{{m}}$",
)

ax.axvline(
    median_position_error.item(),
    color="C3",
    linestyle=":",
    label=rf"Median: ${median_position_error.item():.3f}\,\mathrm{{m}}$",
)

# Add legend-only entries for std and SEM
ax.plot(
    [],
    [],
    linestyle="None",
    label=rf"Std: ${position_error_std.item():.3f}\,\mathrm{{m}}$",
)

ax.plot(
    [],
    [],
    linestyle="None",
    label=rf"SEM: ${position_error_sem.item():.3f}\,\mathrm{{m}}$",
)

ax.set_xlabel(r"Position error $[\mathrm{m}]$")
ax.set_ylabel("Number of events")
ax.set_title("Distribution of Position Reconstruction Error")

# Reorder legend: Mean, Std, SEM, Median
handles, labels = ax.get_legend_handles_labels()
order = [0, 2, 3, 1]
ax.legend(
    [handles[i] for i in order],
    [labels[i] for i in order],
)

fig.tight_layout()
fig.savefig(f"{fig_dir}/position_error_histogram.pdf")
None

In [ ]:
# Get energy values for the test set to check if there is a correlation between the energy of the event and the prediction error.
test_energy = torch.tensor(
    ak.to_numpy(test_dataset["energy"]),
    dtype=torch.float32
)

print(test_energy.shape)

# Position error vs energy
fig, ax = plt.subplots(figsize=(6, 4))

energy_np = test_energy.numpy()

ax.scatter(
    energy_np,
    position_error.numpy(),
    s=6,
    alpha=0.4,
    color="C0",
)

ax.set_xlabel("Energy")
ax.set_ylabel("Position error")
ax.set_title("Position Error vs Energy")

fig.tight_layout()
fig.savefig(f"{fig_dir}/position_error_vs_energy.pdf")
None

In [ ]:
# Try to visualize the detector positions using all dataset splits

datasets = [train_dataset, val_dataset, test_dataset]

all_hit_x = np.concatenate([
    ak.to_numpy(ak.flatten(dataset["data"][:, 1, :]))
    for dataset in datasets
])

all_hit_y = np.concatenate([
    ak.to_numpy(ak.flatten(dataset["data"][:, 2, :]))
    for dataset in datasets
])

# Unique detector positions
detector_positions = np.unique(
    np.column_stack([all_hit_x, all_hit_y]),
    axis=0
)

det_x = detector_positions[:, 0]
det_y = detector_positions[:, 1]

print("Number of unique detector positions:", len(detector_positions))
print(detector_positions)

In [ ]:
# Convert tensors to numpy
true_x_np = true_x.numpy()
true_y_np = true_y.numpy()
pred_x_np = pred_x.numpy()
pred_y_np = pred_y.numpy()

# Euclidean error in meters
position_error_np = np.sqrt((pred_x_np - true_x_np)**2 + (pred_y_np - true_y_np)**2)

# Random subset so the plot stays readable
rng = np.random.default_rng(41)
n_plot = 300
idx = rng.choice(len(true_x_np), size=n_plot, replace=False)

# Build line segments: each line goes from true position to predicted position
segments = np.stack(
    [
        np.column_stack([true_x_np[idx], true_y_np[idx]]),
        np.column_stack([pred_x_np[idx], pred_y_np[idx]])
    ],
    axis=1
)

# Normalize colors to the error range
norm = Normalize(
    vmin=position_error_np[idx].min(),
    vmax=position_error_np[idx].max()
)

norm = PowerNorm(
    gamma=0.7,
    vmin=position_error_np[idx].min(),
    vmax=position_error_np[idx].max()
)

# make larger errors slightly thicker
linewidths = 0.4 + 1.2 * (
    (position_error_np[idx] - position_error_np[idx].min()) /
    (position_error_np[idx].max() - position_error_np[idx].min() + 1e-12)
)

# Create colored line collection
lc = LineCollection(
    segments,
    cmap="inferno",
    norm=norm,
    linewidths=linewidths,
    alpha=0.8,
)
lc.set_array(position_error_np[idx])

fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

# Detector positions in background
ax.scatter(
    det_x,
    det_y,
    s=30,
    color="black",
    alpha=0.8,
    label="Detector positions",
)

# Add colored reconstruction lines
line_handle = ax.add_collection(lc)

# True positions
ax.scatter(
    true_x_np[idx],
    true_y_np[idx],
    s=18,
    color="C0",
    alpha=0.9,
    label="True positions",
)

# Predicted positions
ax.scatter(
    pred_x_np[idx],
    pred_y_np[idx],
    s=18,
    color="C1",
    alpha=0.9,
    label="Predicted positions",
)


ax.set_xlabel(r"x position $[\mathrm{m}]$")
ax.set_ylabel(r"y position $[\mathrm{m}]$")
ax.set_title(f"2D Reconstruction Error of {n_plot} Random Test Events")
ax.set_aspect("equal")

# Colorbar for error magnitude
cbar = fig.colorbar(line_handle, ax=ax)
cbar.set_label(r"Position error $[\mathrm{m}]$")

ax.legend()
fig.savefig(f"{fig_dir}/reconstruction_error_2d_colored.pdf")
None

In [ ]:
# Convert to numpy
true_x_np = true_x.numpy()
true_y_np = true_y.numpy()
position_error_np = position_error.numpy()

# Number of bins
n_bins_x = 8
n_bins_y = 8

# Bin edges based on TRUE positions
x_edges = np.linspace(true_x_np.min(), true_x_np.max(), n_bins_x + 1)
y_edges = np.linspace(true_y_np.min(), true_y_np.max(), n_bins_y + 1)

# Assign each event to a bin based on TRUE position
x_bin = np.digitize(true_x_np, x_edges) - 1
y_bin = np.digitize(true_y_np, y_edges) - 1

# Fix events exactly on the upper edge
x_bin = np.clip(x_bin, 0, n_bins_x - 1)
y_bin = np.clip(y_bin, 0, n_bins_y - 1)

# Arrays to store statistics
mean_error_grid = np.full((n_bins_y, n_bins_x), np.nan)
std_error_grid = np.full((n_bins_y, n_bins_x), np.nan)
count_grid = np.zeros((n_bins_y, n_bins_x), dtype=int)

# Fill the grids
for iy in range(n_bins_y):
    for ix in range(n_bins_x):
        mask = (x_bin == ix) & (y_bin == iy)
        values = position_error_np[mask]

        count_grid[iy, ix] = len(values)

        if len(values) > 0:
            mean_error_grid[iy, ix] = np.mean(values)
            std_error_grid[iy, ix] = np.std(values)

# Optional: require a minimum number of events per bin
min_count = 5
mean_error_grid[count_grid < min_count] = np.nan
std_error_grid[count_grid < min_count] = np.nan

In [ ]:
# Convert to numpy
true_x_np = true_x.numpy()
true_y_np = true_y.numpy()
position_error_np = position_error.numpy()

# Slightly increased number of bins
n_bins_x = 10
n_bins_y = 10

# Bin edges based on TRUE positions
x_edges = np.linspace(true_x_np.min(), true_x_np.max(), n_bins_x + 1)
y_edges = np.linspace(true_y_np.min(), true_y_np.max(), n_bins_y + 1)

# Assign each event to a bin based on TRUE position
x_bin = np.digitize(true_x_np, x_edges) - 1
y_bin = np.digitize(true_y_np, y_edges) - 1

# Fix values on the upper edge
x_bin = np.clip(x_bin, 0, n_bins_x - 1)
y_bin = np.clip(y_bin, 0, n_bins_y - 1)

# Arrays to store statistics
mean_error_grid = np.full((n_bins_y, n_bins_x), np.nan)
std_error_grid = np.full((n_bins_y, n_bins_x), np.nan)
count_grid = np.zeros((n_bins_y, n_bins_x), dtype=int)

# Fill the grids
for iy in range(n_bins_y):
    for ix in range(n_bins_x):
        mask = (x_bin == ix) & (y_bin == iy)
        values = position_error_np[mask]

        count_grid[iy, ix] = len(values)

        if len(values) > 0:
            mean_error_grid[iy, ix] = np.mean(values)
            std_error_grid[iy, ix] = np.std(values)

# hide bins with too few entries
min_count = 5
mean_error_grid[count_grid < min_count] = np.nan
std_error_grid[count_grid < min_count] = np.nan

# Plot
fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)

mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    mean_error_grid,
    cmap="plasma",
    shading="auto",
)

cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label(r"Mean position error $[\mathrm{m}]$")

# Bin centers for text placement
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])

# Put std value inside each populated bin
for iy, yc in enumerate(y_centers):
    sigma_written_in_row = False

    for ix, xc in enumerate(x_centers):
        if not np.isnan(std_error_grid[iy, ix]):

            # Only the first populated bin in each row shows "\sigma="
            if not sigma_written_in_row:
                label_text = rf"$\sigma={std_error_grid[iy, ix]:.2f}$"
                sigma_written_in_row = True
            else:
                label_text = rf"${std_error_grid[iy, ix]:.2f}$"

            ax.text(
                xc,
                yc,
                label_text,
                ha="center",
                va="center",
                color="black",
                fontsize=8,
            )

ax.scatter(
    det_x,
    det_y,
    s=5,
    color="green",
    alpha=0.8,
    label="Detector positions",
)

ax.set_xlabel(r"True x position $[\mathrm{m}]$")
ax.set_ylabel(r"True y position $[\mathrm{m}]$")
ax.set_title("Mean Reconstruction Error by True Position")
ax.legend(loc="upper right")

ax.set_aspect("equal")

fig.savefig(f"{fig_dir}/position_error_heatmap_mean_with_std.pdf")
None